In [1]:
# === ノートブック共通の前処理 (llm_math パッケージの読み込み) ===
import sys
from pathlib import Path

# llm_math パッケージの候補パス
_candidates = [
    '.', 'src', '..', '../src',
    '/content/llm-math-book/src',
    '/content/llm-math-book',
    '/workspace/src',
    '/workspace',
]
# 親ディレクトリも候補に追加 (notebooks/ フォルダで実行する場合)
for p in Path.cwd().parents:
    _candidates.append(str(p / 'src'))
    _candidates.append(str(p))

for p in _candidates:
    if p and p not in sys.path and Path(p).exists():
        sys.path.insert(0, p)

# llm_math の import を試行
try:
    from llm_math import viz, bench, data
    _LLM_MATH_OK = True
except ImportError as e:
    _LLM_MATH_OK = False
    print(f"[注意] llm_math パッケージの読み込み テキスト: {e}")
    print("  GitHub リポジトリを clone して colab_setup.sh を実行してください。")
# === 前処理ここまで ===


# Ch 07. バックプロパゲーション テキスト — テキスト テキスト テキスト ⭐

> **学習目標**
> - テキスト テキスト テキスト 学習テキスト テキスト テキスト テキスト度テキスト
> - 計算 グラフ(computational graph)テキスト テキスト テキスト 導関数テキスト 計算テキスト
> - Autogradテキスト テキスト テキスト テキスト
> - バックプロパゲーションテキスト CPU vs GPU 速度 テキスト テキスト

テキスト テキスト テキスト テキスト テキスト. PyTorchテキスト `autograd`テキスト テキスト テキスト テキスト テキスト テキスト テキスト.

## 7.1 テキスト バックプロパゲーションテキスト

テキスト テキスト $\mathcal{L}(\theta)$テキスト テキスト テキスト $\nabla_\theta \mathcal{L}$テキスト テキスト.

**テキスト方向 数値微分(forward-mode AD)**: テキスト テキスト テキスト テキスト 順伝播.
- テキスト $N$テキスト → $N$テキスト 順伝播. テキスト.

**テキスト方向 テキスト テキスト(reverse-mode AD, バックプロパゲーション)**: テキスト テキスト バックプロパゲーションテキスト テキスト テキスト テキスト 計算.
- $N$テキスト テキスト → 1テキスト 順伝播 + 1テキスト バックプロパゲーション. テキスト度テキスト テキスト.


## 7.2 テキスト テキスト テキスト

$$\frac{d}{dx} f(g(x)) = f'(g(x)) \cdot g'(x)$$

テキスト テキスト: $y = f(u_1, u_2, \ldots, u_n)$, $u_i = g_i(x)$テキスト テキスト

$$\frac{dy}{dx} = \sum_i \frac{\partial f}{\partial u_i} \cdot \frac{du_i}{dx}$$

テキスト バックプロパゲーションテキスト テキスト テキスト.


In [2]:
import numpy as np
import matplotlib.pyplot as plt

# テキスト テキスト 関数: y = f(g(h(x))) = (h(x)^2 + 1) * 2, h(x) = x + 1
# = ((x+1)^2 + 1) * 2
def f(x): return ((x + 1)**2 + 1) * 2
def df_dx(x): return 2 * (x + 1) * 2 * 1  # テキスト テキスト

# 数値微分テキスト 検証
def numerical_diff(f, x, h=1e-5):
    return (f(x + h) - f(x - h)) / (2 * h)

x0 = 3.0
print(f"f(x) = ((x+1)^2 + 1) * 2")
print(f"f'({x0}) 解析的 (テキスト テキスト): {df_dx(x0)}")
print(f"f'({x0}) テキスト: {numerical_diff(f, x0):.6f}")
print(f"Error: {abs(df_dx(x0) - numerical_diff(f, x0)):.2e}")


f(x) = ((x+1)^2 + 1) * 2
f'(3.0) 解析的 (テキスト テキスト): 16.0
f'(3.0) テキスト: 16.000000
Error: 2.50e-10


## 7.3 計算 グラフ

テキスト 関数テキスト テキスト テキスト(テキスト, テキスト, テキスト テキスト)テキスト グラフテキスト テキスト.

テキスト: $\mathcal{L} = (\sigma(w \cdot x + b) - y)^2$

```
x ──┐
    ├─→ (×) ──→ (+) ──→ (σ) ──→ (-) ──→ (²) ──→ L
w ──┘         ↑              ↑
              b              y
```

テキスト テキスト **テキスト 導関数(local gradient)**テキスト 計算テキスト テキスト. バックプロパゲーションテキスト テキスト テキスト テキスト テキスト.


In [3]:
# テキスト Autograd テキスト テキスト
class Tensor:
    """テキスト テキスト Differential Tensor."""
    def __init__(self, data, requires_grad=False, _children=(), _op=''):
        self.data = np.asarray(data, dtype=float)
        self.requires_grad = requires_grad
        self.grad = None
        self._backward = lambda: None
        self._prev = set(_children)
        self._op = _op

    def __repr__(self):
        return f"Tensor({self.data}, grad={self.grad})"

    @property
    def shape(self):
        return self.data.shape

    # Operation テキスト
    def __add__(self, other):
        other = other if isinstance(other, Tensor) else Tensor(other)
        out = Tensor(self.data + other.data, _children=(self, other), _op='+')

        def _backward():
            # Additionテキスト テキスト Derivative = 1
            self.grad = (self.grad if self.grad is not None else 0) + out.grad
            other.grad = (other.grad if other.grad is not None else 0) + out.grad
        out._backward = _backward
        return out

    def __mul__(self, other):
        other = other if isinstance(other, Tensor) else Tensor(other)
        out = Tensor(self.data * other.data, _children=(self, other), _op='*')

        def _backward():
            # Multiplicationテキスト テキスト Derivative: d/da(a*b) = b, d/db(a*b) = a
            self.grad = (self.grad if self.grad is not None else 0) + other.data * out.grad
            other.grad = (other.grad if other.grad is not None else 0) + self.data * out.grad
        out._backward = _backward
        return out

    def relu(self):
        out = Tensor(np.maximum(0, self.data), _children=(self,), _op='relu')

        def _backward():
            self.grad = (self.grad if self.grad is not None else 0) + (self.data > 0) * out.grad
        out._backward = _backward
        return out

    def sigmoid(self):
        s = 1 / (1 + np.exp(-self.data))
        out = Tensor(s, _children=(self,), _op='sigmoid')

        def _backward():
            self.grad = (self.grad if self.grad is not None else 0) + s * (1 - s) * out.grad
        out._backward = _backward
        return out

    def sum(self):
        out = Tensor(self.data.sum(), _children=(self,), _op='sum')

        def _backward():
            self.grad = (self.grad if self.grad is not None else 0) + np.ones_like(self.data) * out.grad
        out._backward = _backward
        return out

    def backward(self):
        # テキスト Alignment
        topo = []
        visited = set()
        def build(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build(child)
                topo.append(v)
        build(self)

        self.grad = np.ones_like(self.data)  # dL/dL = 1
        for v in reversed(topo):
            v._backward()

# Test: f(x, y) = (x + y) * (x - y) = x^2 - y^2 (テキスト x*x - y*y)
x = Tensor(3.0, requires_grad=True)
y = Tensor(4.0, requires_grad=True)
# z = x*x - y*y  (subtraction = add negative; Implementation テキスト テキスト テキスト)
a = x * x  # x^2
b = y * y  # y^2
z = a + (b * Tensor(-1.0))  # x^2 - y^2
z.backward()

print(f"x = {x.data}, y = {y.data}")
print(f"z = x^2 - y^2 = {z.data}")
print(f"dz/dx = {x.grad} (Theory 2x = {2*x.data})")
print(f"dz/dy = {y.grad} (Theory -2y = {-2*y.data})")


x = 3.0, y = 4.0
z = x^2 - y^2 = -7.0
dz/dx = 6.0 (Theory 2x = 6.0)
dz/dy = -8.0 (Theory -2y = -8.0)


## 7.4 テキスト バックプロパゲーション テキスト度

テキスト: $\mathbf{y} = W\mathbf{x} + \mathbf{b}$

テキスト $\mathcal{L}$テキスト テキスト テキスト:

$$\frac{\partial \mathcal{L}}{\partial W} = \frac{\partial \mathcal{L}}{\partial \mathbf{y}} \cdot \mathbf{x}^\top = \boldsymbol{\delta} \, \mathbf{x}^\top$$

$$\frac{\partial \mathcal{L}}{\partial \mathbf{x}} = W^\top \boldsymbol{\delta}$$

$$\frac{\partial \mathcal{L}}{\partial \mathbf{b}} = \boldsymbol{\delta}$$

テキスト $\boldsymbol{\delta} = \frac{\partial \mathcal{L}}{\partial \mathbf{y}}$テキスト "テキスト テキスト".


In [4]:
# テキスト バックプロパゲーション 検証 (PyTorchテキスト 比較)
import torch

# テキスト
x_np = np.random.randn(3, 4)  # batch=3, in=4
W_np = np.random.randn(4, 5)  # in=4, out=5
b_np = np.random.randn(5)

# NumPyテキスト 順伝播 + バックプロパゲーション
def linear_forward(x, W, b):
    return x @ W + b

def linear_backward(x, W, delta):
    """delta: dL/dy (N, out)"""
    dW = x.T @ delta          # (in, out)
    db = delta.sum(axis=0)    # (out,)
    dx = delta @ W.T          # (N, in)
    return dW, db, dx

# テキスト delta (dL/dy)
delta_np = np.random.randn(3, 5)
y_np = linear_forward(x_np, W_np, b_np)
dW_np, db_np, dx_np = linear_backward(x_np, W_np, delta_np)

# PyTorchテキスト Validation
x_t = torch.tensor(x_np, requires_grad=True)
W_t = torch.tensor(W_np, requires_grad=True)
b_t = torch.tensor(b_np, requires_grad=True)
y_t = x_t @ W_t + b_t
y_t.backward(torch.tensor(delta_np))

print("NumPy vs PyTorch バックプロパゲーション 比較:")
print(f"  dW Maximum Error: {np.max(np.abs(dW_np - W_t.grad.numpy())):.2e}")
print(f"  db Maximum Error: {np.max(np.abs(db_np - b_t.grad.numpy())):.2e}")
print(f"  dx Maximum Error: {np.max(np.abs(dx_np - x_t.grad.numpy())):.2e}")
print("\n=> NumPy テキスト Implementationテキスト PyTorch autogradテキスト テキスト テキスト!")


NumPy vs PyTorch バックプロパゲーション 比較:
  dW Maximum Error: 0.00e+00
  db Maximum Error: 0.00e+00
  dx Maximum Error: 8.88e-16

=> NumPy テキスト Implementationテキスト PyTorch autogradテキスト テキスト テキスト!


## 7.5 MLP バックプロパゲーション テキスト テキスト度

2テキスト MLP: $\mathbf{o} = W_2 \sigma(W_1 \mathbf{x} + \mathbf{b}_1) + \mathbf{b}_2$

テキスト $\mathcal{L} = \|\mathbf{o} - \mathbf{y}\|^2$ (MSE)

バックプロパゲーション:
1. $\frac{\partial \mathcal{L}}{\partial \mathbf{o}} = 2(\mathbf{o} - \mathbf{y})$
2. $\frac{\partial \mathcal{L}}{\partial W_2} = \frac{\partial \mathcal{L}}{\partial \mathbf{o}} \mathbf{h}^\top$
3. $\frac{\partial \mathcal{L}}{\partial \mathbf{h}} = W_2^\top \frac{\partial \mathcal{L}}{\partial \mathbf{o}}$
4. $\frac{\partial \mathcal{L}}{\partial \mathbf{z}_1} = \frac{\partial \mathcal{L}}{\partial \mathbf{h}} \odot \sigma'(\mathbf{z}_1)$
5. $\frac{\partial \mathcal{L}}{\partial W_1} = \frac{\partial \mathcal{L}}{\partial \mathbf{z}_1} \mathbf{x}^\top$

テキスト: **テキスト テキスト テキスト 導関数 × テキスト テキスト**テキスト テキスト.


In [5]:
# MLP バックプロパゲーション 数値微分テキスト 検証
# f(W1, b1, W2, b2) = ||W2 @ sigmoid(W1 @ x + b1) + b2 - y||^2

def mlp_loss_and_grads(x, W1, b1, W2, b2, y):
    """Forward Pass + Backward Pass (Vector Input, テキスト Sample)."""
    # Forward Pass
    z1 = W1 @ x + b1
    h = 1 / (1 + np.exp(-z1))  # sigmoid
    o = W2 @ h + b2
    diff = o - y
    loss = np.sum(diff**2)
    # Backward Pass
    do = 2 * diff
    dW2 = np.outer(do, h)
    db2 = do
    dh = W2.T @ do
    dz1 = dh * h * (1 - h)
    dW1 = np.outer(dz1, x)
    db1 = dz1
    return loss, dW1, db1, dW2, db2

# テキスト Differential
def numerical_grad(f, param, h=1e-5):
    """fテキスト lossテキスト テキスト Function. paramテキスト テキスト テキスト Gradient."""
    grad = np.zeros_like(param)
    it = np.nditer(param, flags=['multi_index'])
    while not it.finished:
        idx = it.multi_index
        old = param[idx]
        param[idx] = old + h
        loss_plus = f()
        param[idx] = old - h
        loss_minus = f()
        param[idx] = old
        grad[idx] = (loss_plus - loss_minus) / (2 * h)
        it.iternext()
    return grad

# テキスト MLPテキスト Validation
np.random.seed(0)
x = np.random.randn(3)
y = np.random.randn(2)
W1 = np.random.randn(4, 3)
b1 = np.random.randn(4)
W2 = np.random.randn(2, 4)
b2 = np.random.randn(2)

loss, dW1, db1, dW2, db2 = mlp_loss_and_grads(x, W1, b1, W2, b2, y)

# テキスト Gradient
loss_fn = lambda: mlp_loss_and_grads(x, W1, b1, W2, b2, y)[0]
dW1_num = numerical_grad(loss_fn, W1)
dW2_num = numerical_grad(loss_fn, W2)

print(f"dW1 テキスト 誤差: {np.max(np.abs(dW1 - dW1_num)):.2e}")
print(f"dW2 Maximum Error: {np.max(np.abs(dW2 - dW2_num)):.2e}")
print("=> 解析的 Backward Passテキスト テキスト Differentialテキスト テキスト!")


dW1 テキスト 誤差: 4.81e-11
dW2 Maximum Error: 2.04e-11
=> 解析的 Backward Passテキスト テキスト Differentialテキスト テキスト!


## 7.6 テキスト テキスト/テキスト

テキスト 導関数 $\sigma'(x) = \sigma(x)(1-\sigma(x)) \leq 0.25$.

テキスト テキスト バックプロパゲーション テキスト テキスト テキスト 値テキスト テキスト テキスト テキスト テキスト テキスト → **テキスト テキスト(vanishing gradient)**.

テキスト:
- ReLU テキスト テキスト (導関数テキスト 0 テキスト 1)
- テキスト テキスト テキスト (He, Xavier)
- Residual connection (テキスト テキスト)
- BatchNorm/LayerNorm


In [6]:
# テキスト テキスト テキスト
def sigmoid(x): return 1 / (1 + np.exp(-x))
def sigmoid_deriv(x):
    s = sigmoid(x)
    return s * (1 - s)

# テキスト テキスト テキスト テキスト
np.random.seed(0)
depths = [5, 10, 20, 50, 100]
print(f"{'Depth':>6} | {'Mean |grad|':>15} | {'Minimum |grad|':>15}")
print("-" * 45)
for d in depths:
    x = np.random.randn(100)
    grad = np.ones(100)  # dL/dy
    for _ in range(d):
        # Backward Pass テキスト Layer
        z = np.random.randn(100)
        grad = grad * sigmoid_deriv(z)
    print(f"{d:>6} | {np.mean(np.abs(grad)):>15.2e} | {np.min(np.abs(grad)):>15.2e}")

print("\n=> Depthテキスト テキスト Gradientテキスト テキスト テキスト (テキスト)!")
print("ReLUテキスト テキストPlane Derivativeテキスト 0 テキスト 1テキスト テキスト 問題テキスト テキスト テキスト.")


 Depth |     Mean |grad| |  Minimum |grad|
---------------------------------------------
     5 |        3.71e-04 |        2.23e-05
    10 |        1.51e-07 |        1.51e-08
    20 |        2.49e-14 |        5.22e-16
    50 |        6.93e-35 |        2.79e-38
   100 |        2.70e-69 |        2.82e-73

=> Depthテキスト テキスト Gradientテキスト テキスト テキスト (テキスト)!
ReLUテキスト テキストPlane Derivativeテキスト 0 テキスト 1テキスト テキスト 問題テキスト テキスト テキスト.


## 7.7 [CPU/GPU ベンチマーク ②] バックプロパゲーション 時間 比較

モデルサイズ別テキスト 順伝播+バックプロパゲーション 時間テキスト CPU vs GPUテキスト 比較テキスト.


In [7]:
# バックプロパゲーション ベンチマーク
import torch
import time
from llm_math.bench import time_fn, format_results_table

def make_mlp_and_run(input_dim, hidden, output_dim, n_layers, batch_size=32, device='cpu'):
    """MLPテキスト テキスト テキスト Step Forward Pass+Backward Pass."""
    dims = [input_dim] + [hidden] * (n_layers - 1) + [output_dim]
    layers = [torch.nn.Linear(dims[i], dims[i+1]) for i in range(len(dims) - 1)]
    model = torch.nn.Sequential(*layers).to(device)
    x = torch.randn(batch_size, input_dim, device=device)
    y = torch.randn(batch_size, output_dim, device=device)
    loss_fn = torch.nn.MSELoss()
    opt = torch.optim.SGD(model.parameters(), lr=0.01)

    def step():
        opt.zero_grad()
        out = model(x)
        loss = loss_fn(out, y)
        loss.backward()
        opt.step()
        return loss
    return step

# Model Sizeテキスト Comparison
configs = [
    ('small',  100, 64, 10, 3),
    ('medium', 100, 256, 10, 5),
    ('large',  100, 512, 10, 10),
    ('xlarge', 100, 1024, 10, 20),
]

print(f"{'Config':>8} | {'CPU (ms)':>12} | {'GPU (ms)':>12} | {'Speedup':>10}")
print("-" * 50)
for name, in_d, hid, out_d, nl in configs:
    step_cpu = make_mlp_and_run(in_d, hid, out_d, nl, device='cpu')
    res_cpu = time_fn(step_cpu, device='cpu', warmup=2, repeat=5)
    if torch.cuda.is_available():
        step_gpu = make_mlp_and_run(in_d, hid, out_d, nl, device='cuda')
        res_gpu = time_fn(step_gpu, device='cuda', warmup=3, repeat=5)
        speedup = res_cpu['mean_ms'] / res_gpu['mean_ms']
        print(f"{name:>8} | {res_cpu['mean_ms']:>12.3f} | {res_gpu['mean_ms']:>12.3f} | {speedup:>9.2f}x")
    else:
        print(f"{name:>8} | {res_cpu['mean_ms']:>12.3f} | {'N/A':>12} | {'N/A':>10}")

if not torch.cuda.is_available():
    print("\n⚠ GPU テキスト. Colabテキスト T4 GPU テキスト テキストPlane テキスト テキスト テキスト テキスト.")
    print("  テキスト テキスト Model(xlarge)テキスト CPUテキスト テキスト テキスト, GPUテキスト テキスト msテキスト テキスト.")


  Config |     CPU (ms) |     GPU (ms) |    Speedup
--------------------------------------------------


   small |        0.341 |          N/A |        N/A
  medium |        3.122 |          N/A |        N/A
   large |       13.843 |          N/A |        N/A


  xlarge |       90.379 |          N/A |        N/A

⚠ GPU テキスト. Colabテキスト T4 GPU テキスト テキストPlane テキスト テキスト テキスト テキスト.
  テキスト テキスト Model(xlarge)テキスト CPUテキスト テキスト テキスト, GPUテキスト テキスト msテキスト テキスト.


## 7.8 要点

| テキスト | テキスト | テキスト |
|---|---|---|
| テキスト テキスト | $\frac{d}{dx}f(g(x)) = f'(g)g'(x)$ | テキスト関数 テキスト |
| 計算 グラフ | — | テキスト関数テキスト テキスト テキスト テキスト |
| テキスト 導関数 | — | テキスト テキスト テキスト テキスト 出力 テキスト |
| バックプロパゲーション | $\nabla_\theta \mathcal{L}$ | 1テキスト バックプロパゲーションテキスト テキスト テキスト |
| テキスト バックプロパゲーション | $\partial\mathcal{L}/\partial W = \delta \mathbf{x}^\top$ | テキスト テキスト |

## 演習問題

1. テキスト Autograd `Tensor` テキスト `__sub__`, `__truediv__`, `matmul` テキスト テキスト.
2. テキスト Autogradテキスト $f(x, y) = (x + y)^2 - xy$テキスト $\partial f/\partial x, \partial f/\partial y$テキスト 計算テキスト 数値微分テキスト 検証テキスト.
3. PyTorchテキスト 3テキスト MLPテキスト テキスト, テキスト テキスト バックプロパゲーション テキスト `W1.grad`, `W2.grad`テキスト 出力テキスト.
4. テキスト 1, 5, 10, 20テキスト MLPテキスト gradient normテキスト 比較テキスト テキスト テキスト テキスト テキスト.
5. He テキスト テキスト テキスト(テキスト `randn`)テキスト 比較テキスト テキスト MLP 学習テキスト テキスト テキスト テキスト.

> 解答: `solutions/ch07_solutions.ipynb`
